In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"

In [2]:
ist_now      = datetime.now(ZoneInfo("Asia/Kolkata"))
current_hour = ist_now.hour

print(f"Current IST Time  : {ist_now.strftime('%I:%M:%S %p')}")
print(f"Chart window      : 6:00 PM – 9:00 PM IST")
print(f"Access granted    : {18 <= current_hour < 21}")

Current IST Time  : 07:25:38 PM
Chart window      : 6:00 PM – 9:00 PM IST
Access granted    : True


In [3]:
np.random.seed(42)

all_categories = [
    'EDUCATION','ENTERTAINMENT','EVENTS',        # starts with E
    'BEAUTY','BUSINESS',                          # starts with B
    'COMICS','COMMUNICATION',                     # starts with C
    'GAME','TOOLS','FAMILY','FINANCE'             # other categories
]

# App name word pool — no words containing letter S
word_pool = [
    'Nova','Bolt','Fit','Zen','Glow','Peak','Flow','Flip','Grid','Klip',
    'Zap','Wink','Vault','Mint','Core','Drift','Bright','Punch','Echo','Link',
    'Tide','Orbit','Comet','Nimble','Vivid','Claro','Luxe','Forma','Rapid','Prime',
    'Clarity','Aurora','Beacon','Cedar','Delta','Ember','Flora','Haven','Igloo','Juniper',
    'Karma','Lumen','Marvel','Notion','Optic','Pearl','Quarry','Relay','Terra','Unity',
    'Verve','Wagon','Alpha','Bravo','Charlie','Delta','Echo','Foxtrot','Golf','Hotel'
]

# Monthly date range: Jan 2021 to Dec 2023 = 36 months
months = pd.date_range(start='2021-01-01', periods=36, freq='MS')

rows = []
for month in months:
    for cat in all_categories:
        n_apps = np.random.randint(15, 40)
        for _ in range(n_apps):
            # Generate app name from 1–2 words
            name = ' '.join(np.random.choice(word_pool, size=np.random.randint(1, 3)))
            installs = int(np.random.choice(
                [1000, 5000, 10000, 50000, 100000, 500000, 1000000],
                p=[0.05, 0.10, 0.20, 0.25, 0.20, 0.12, 0.08]
            ))
            reviews = np.random.randint(0, 5000)
            rows.append({
                'App'      : name,
                'Category' : cat,
                'Installs' : installs,
                'Reviews'  : reviews,
                'Month'    : month
            })

raw_df = pd.DataFrame(rows)
print(f"Raw dataset rows  : {len(raw_df)}")
print(f"Months covered    : {raw_df['Month'].min().strftime('%b %Y')} → {raw_df['Month'].max().strftime('%b %Y')}")
print(f"Categories        : {sorted(raw_df['Category'].unique().tolist())}")

Raw dataset rows  : 10562
Months covered    : Jan 2021 → Dec 2023
Categories        : ['BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION', 'EDUCATION', 'ENTERTAINMENT', 'EVENTS', 'FAMILY', 'FINANCE', 'GAME', 'TOOLS']


In [4]:
df = raw_df.copy()

# ── Filter 1: App name must NOT start with X, Y, or Z ──
# str[0] gets first character → .str.upper() makes it case-insensitive
df = df[~df['App'].str[0].str.upper().isin(['X', 'Y', 'Z'])]
print(f"After removing apps starting with X/Y/Z : {len(df)}")

# ── Filter 2: App name must NOT contain letter 'S' anywhere ──
# case=False checks both uppercase S and lowercase s
df = df[~df['App'].str.contains('s', case=False, na=False)]
print(f"After removing apps containing letter S : {len(df)}")

# ── Filter 3: Reviews must be more than 500 ──
df = df[df['Reviews'] > 500]
print(f"After filtering reviews > 500           : {len(df)}")

# ── Filter 4: Category must start with E, C, or B only ──
df = df[df['Category'].str.upper().str[0].isin(['E', 'C', 'B'])]
print(f"After keeping only E/C/B categories     : {len(df)}")
print(f"Categories remaining                    : {sorted(df['Category'].unique().tolist())}")

After removing apps starting with X/Y/Z : 10216
After removing apps containing letter S : 10216
After filtering reviews > 500           : 9219
After keeping only E/C/B categories     : 5922
Categories remaining                    : ['BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION', 'EDUCATION', 'ENTERTAINMENT', 'EVENTS']


In [5]:
translation_map = {
    'BEAUTY'   : 'BEAUTY (सौंदर्य)',     # Hindi translation
    'BUSINESS' : 'BUSINESS (வணிகம்)',    # Tamil translation
}

# map() applies translation where available, fillna keeps originals unchanged
df['Category_Display'] = df['Category'].map(translation_map).fillna(df['Category'])

print("Category display names after translation:")
print("-" * 45)
for orig, display in df.groupby('Category')['Category_Display'].first().items():
    print(f"  {orig:20s} → {display}")
print("-" * 45)

Category display names after translation:
---------------------------------------------
  BEAUTY               → BEAUTY (सौंदर्य)
  BUSINESS             → BUSINESS (வணிகம்)
  COMICS               → COMICS
  COMMUNICATION        → COMMUNICATION
  EDUCATION            → EDUCATION
  ENTERTAINMENT        → ENTERTAINMENT
  EVENTS               → EVENTS
---------------------------------------------


In [6]:
monthly = (
    df.groupby(['Month', 'Category_Display'])['Installs']
    .sum()
    .reset_index()
    .sort_values(['Category_Display', 'Month'])
)

# pct_change() computes (current - previous) / previous * 100 per category
monthly['MoM_Growth'] = (
    monthly.groupby('Category_Display')['Installs']
    .pct_change() * 100
)

# Flag months where growth exceeds 20%
monthly['High_Growth'] = monthly['MoM_Growth'] > 20

print("Monthly aggregation sample (first 12 rows):")
print(monthly[['Month','Category_Display','Installs',
               'MoM_Growth','High_Growth']].head(12).to_string(index=False))
print(f"\nTotal months with >20% MoM growth : {monthly['High_Growth'].sum()}")

Monthly aggregation sample (first 12 rows):
     Month Category_Display  Installs  MoM_Growth  High_Growth
2021-01-01 BEAUTY (सौंदर्य)   1600000         NaN        False
2021-02-01 BEAUTY (सौंदर्य)   3380000  111.250000         True
2021-03-01 BEAUTY (सौंदर्य)   3858000   14.142012        False
2021-04-01 BEAUTY (सौंदर्य)   4980000   29.082426         True
2021-05-01 BEAUTY (सौंदर्य)   5192000    4.257028        False
2021-06-01 BEAUTY (सौंदर्य)   3296000  -36.517720        False
2021-07-01 BEAUTY (सौंदर्य)   5248000   59.223301         True
2021-08-01 BEAUTY (सौंदर्य)   3010000  -42.644817        False
2021-09-01 BEAUTY (सौंदर्य)   9206000  205.847176         True
2021-10-01 BEAUTY (सौंदर्य)   2951000  -67.944819        False
2021-11-01 BEAUTY (सौंदर्य)   3941000   33.547950         True
2021-12-01 BEAUTY (सौंदर्य)   1711000  -56.584623        False

Total months with >20% MoM growth : 102


In [11]:
if 18 <= current_hour < 21:

    categories_list = sorted(monthly['Category_Display'].unique())

    # One distinct color per category
    color_map = {
        'BEAUTY (सौंदर्य)'   : '#f472b6',   # Pink
        'BUSINESS (வணிகம்)'  : '#fbbf24',   # Gold
        'COMICS'             : '#34d399',   # Green
        'COMMUNICATION'      : '#60a5fa',   # Blue
        'EDUCATION'          : '#a78bfa',   # Purple
        'ENTERTAINMENT'      : '#fb923c',   # Orange
        'EVENTS'             : '#38bdf8',   # Light Blue
    }

    fig = go.Figure()

    for cat in categories_list:
        cat_df = monthly[monthly['Category_Display'] == cat].sort_values('Month')
        color  = color_map.get(cat, '#94a3b8')
        x_vals = cat_df['Month'].tolist()
        y_vals = cat_df['Installs'].tolist()
        growth = cat_df['High_Growth'].tolist()

        # ── Trace 1: Main line with markers ──────────────────
        fig.add_trace(go.Scatter(
            x             = x_vals,
            y             = y_vals,
            name          = cat,
            mode          = 'lines+markers',
            line          = dict(color=color, width=2),
            marker        = dict(size=5, color=color),
            hovertemplate = (
                f'<b>{cat}</b><br>'
                'Month: %{x|%b %Y}<br>'
                'Total Installs: %{y:,.0f}<br>'
                'MoM Growth: %{customdata:.1f}%<extra></extra>'
            ),
            customdata    = cat_df['MoM_Growth'].fillna(0).tolist()
        ))

        # ── Trace 2: Star markers on high growth months ──────
        # Filters only the months where MoM growth exceeded 20%
        high_x = [x for x, h in zip(x_vals, growth) if h]
        high_y = [y for y, h in zip(y_vals, growth) if h]

        if high_x:
            fig.add_trace(go.Scatter(
                x             = high_x,
                y             = high_y,
                name          = f'{cat} >20% Growth',
                mode          = 'markers',
                marker        = dict(
                    size      = 14,
                    color     = color,
                    symbol    = 'star',
                    opacity   = 0.95,
                    line      = dict(color='white', width=1)
                ),
                showlegend    = False,
                hovertemplate = (
                    f'<b> HIGH GROWTH — {cat}</b><br>'
                    'Month: %{x|%b %Y}<br>'
                    'Installs: %{y:,.0f}<br>'
                    '<i>Month-over-Month growth > 20%</i><extra></extra>'
                )
            ))

    # ── Vertical shading for all high-growth months ───────────
    # Adds a faint yellow band across all categories for that month
    high_months = monthly[monthly['High_Growth']]['Month'].unique()
    for hm in high_months:
        fig.add_vrect(
            x0        = hm,
            x1        = pd.Timestamp(hm) + pd.offsets.MonthEnd(1),
            fillcolor = 'rgba(255, 255, 100, 0.05)',
            line_width= 0,
            layer     = 'below'
        )

    # ── Layout ─────────────────────────────────────────────────
    fig.update_layout(
        title = dict(
            text    = ('Total Installs Trend Over Time — by App Category<br>'
                       '<sup>Filters: Name not starting X/Y/Z | No "S" in name | '
                       'Reviews > 500 | Categories E/C/B | '
                       ' = >20% MoM Growth | 6PM–9PM IST</sup>'),
            font    = dict(size=13, color='#e8eaf0'),
            x       = 0.5,
            xanchor = 'center'
        ),
        paper_bgcolor = '#0f1117',
        plot_bgcolor  = '#161b26',

        xaxis = dict(
            title      = dict(text='Month', font=dict(color='#8892a4', size=12)),
            tickfont   = dict(color='#8892a4', size=10),
            gridcolor  = '#1e2435',
            tickformat = '%b %Y',
            tickangle  = -35,
            showgrid   = True
        ),

        yaxis = dict(
            title     = dict(text='Total Installs', font=dict(color='#8892a4', size=12)),
            tickfont  = dict(color='#8892a4', size=10),
            gridcolor = '#1e2435',
            tickformat= ',.0f',
            zeroline  = False
        ),

        legend = dict(
            bgcolor     = '#1e2435',
            bordercolor = '#252c3d',
            borderwidth = 1,
            font        = dict(color='#c9d1d9', size=10),
            orientation = 'v',
            x           = 1.01,
            xanchor     = 'left',
            y           = 1.0,
            yanchor     = 'top'
        ),

        annotations = [
            dict(
                text      = 'Star markers = months where installs grew > 20% vs previous month  |  Shaded bands = high growth periods',
                x         = 0.5, xref='paper',
                y         = -0.17, yref='paper',
                showarrow = False,
                font      = dict(color='#fbbf24', size=10),
                xanchor   = 'center'
            ),
            dict(
                text      = 'BEAUTY = सौंदर्य (Hindi)  |   BUSINESS = வணிகம் (Tamil)',
                x         = 0.5, xref='paper',
                y         = -0.22, yref='paper',
                showarrow = False,
                font      = dict(color='#94a3b8', size=10),
                xanchor   = 'center'
            )
        ],

        margin    = dict(l=60, r=170, t=110, b=100),
        height    = 600,
        font      = dict(family='Inter, sans-serif'),
        hovermode = 'x unified'
    )

    fig.show()
    print("\n Time series line chart rendered successfully!")
    print(f"   Categories plotted     : {categories_list}")
    print(f"   High growth months     : {len(high_months)}")
    print(f"   Total data points      : {len(monthly)}")

else:
    print("=" * 60)
    print("⏱  CHART ACCESS RESTRICTED")
    print("=" * 60)
    print(f"  This chart is only available : 6:00 PM – 9:00 PM IST")
    print(f"  Your current IST time        : {ist_now.strftime('%I:%M:%S %p')}")
    print(f"  Next window opens            : Today/Tomorrow at 6:00 PM IST")
    print("=" * 60)


 Time series line chart rendered successfully!
   Categories plotted     : ['BEAUTY (सौंदर्य)', 'BUSINESS (வணிகம்)', 'COMICS', 'COMMUNICATION', 'EDUCATION', 'ENTERTAINMENT', 'EVENTS']
   High growth months     : 33
   Total data points      : 252
